# Tutorial 01: Wasserstein Causal Forests (WCF) — Synthetic Quickstart

This notebook walks through **Wasserstein Causal Forests (WCF)** end to end on a synthetic distributional causal inference problem:
1. Representing each unit $i$ by a $K$-grid quantile curve $Q_i \in \mathcal{Q}_K$ on the uniform midpoint grid $u_k = (k - 0.5)/K$ ($K=25$).
2. Declaring distributional functionals (mean, standard deviation, skewness, upper-half mean, $p_{10}$, $p_{90}$, and reference Wasserstein distance $d_{W,K}(q, q_\star^K)$ to an external benchmark law $q_\star^K$).
3. Fitting `DRCalibratedCWDB` ($M$-particle energy-score boosting on the monotone cone $\mathcal{Q}_K$ with arm-shared trees, out-of-fold contrast shrinkage $\lambda^\star \in \{0, 50, 500\}$, and 3-fold AIPW calibration).
4. Predicting conditional particle laws $\widehat{P}_{a,M}^K(x)$ and conditional barycentric quantile curves $\bar{q}_{a,M}^K(x)$.
5. Inspecting doubly robust marginal TATE and moderator-quartile TCATE estimates with influence-function standard errors.

In [ ]:
import os
import sys
from pathlib import Path

if not Path("src/wasserstein_causal_forests").exists() and Path("../src/wasserstein_causal_forests").exists():
    os.chdir("..")
if Path("src").exists() and str(Path("src").resolve()) not in sys.path:
    sys.path.insert(0, str(Path("src").resolve()))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm

from wasserstein_causal_forests import (
    DRCalibratedCWDB,
    ecdf_moderator,
    grid_weights,
    make_functionals,
    midpoint_levels,
    plugin_contrasts,
)

rng = np.random.default_rng(42)
n, n_grid = 300, 25
u = midpoint_levels(n_grid)
z = norm.ppf(u)

raw_x = rng.normal(size=(n, 4))
X = raw_x.copy()
X[:, 0] = ecdf_moderator(raw_x[:, 0])

logits = 0.55 * X[:, 0] - 0.35 * X[:, 1] + 0.25 * np.sin(X[:, 2])
propensity = 1.0 / (1.0 + np.exp(-logits))
A = rng.binomial(1, propensity, size=n).astype(np.int64)

xi_loc = rng.normal(0.0, 0.28, size=n)
xi_scale = np.exp(rng.normal(0.0, 0.14, size=n))
mu_0 = 0.30 * X[:, 0] - 0.25 * X[:, 1] + 0.15 * X[:, 2]
tau_loc = 0.45 + 0.30 * (X[:, 0] > 0.0).astype(float)
sigma_0 = 1.10 + 0.15 * np.abs(X[:, 1])
sigma_1 = 0.88 + 0.10 * np.abs(X[:, 1])

loc = mu_0 + A * tau_loc + xi_loc
scale = ((1 - A) * sigma_0 + A * sigma_1) * xi_scale
Q = np.maximum.accumulate(loc[:, None] + scale[:, None] * z[None, :], axis=1)
q_star = 1.00 + 0.85 * z
weights = grid_weights(n_grid)

print(f"Simulated n={n} units (treated={int(A.sum())}, control={int((A == 0).sum())}), K={n_grid}")

In [ ]:
functionals = make_functionals(q_star, weights)
idx_p10 = int(np.argmin(np.abs(u - 0.10)))
idx_p90 = int(np.argmin(np.abs(u - 0.90)))
functionals["p10"] = lambda q, k=idx_p10: np.asarray(q, dtype=float)[:, k]
functionals["p90"] = lambda q, k=idx_p90: np.asarray(q, dtype=float)[:, k]

model = DRCalibratedCWDB(
    functionals=functionals,
    contrast_candidates=(0.0, 50.0, 500.0),
    n_folds=3,
    n_particles=10,
    n_estimators=60,
    learning_rate=0.12,
    max_depth=4,
    min_samples_leaf=10,
    min_arm_leaf=5,
    collision_epsilon=1e-3,
    arm_shrinkage=5.0,
    architecture="v1",
    sharing="partial",
    init_sharing="pooled",
    random_state=0,
)
model.fit(X, A, Q, weights)

plugin = plugin_contrasts(model, X, functionals)
print(f"Selected contrast shrinkage lambda*: {model.selected_contrast_shrinkage_:.1f}")
print(f"{'Functional':<14} {'WCF (DR)':>10} {'IF SE':>10} {'Plug-in':>10}")
print("-" * 48)
for name in ("mean", "sd", "skew", "tail", "reference", "p10", "p90"):
    print(
        f"{name:<14} {model.dr_marginal(name):+10.3f} "
        f"{model.dr_if_se(name):10.3f} {plugin[name]['marginal']:+10.3f}"
    )

In [ ]:
output_file = "tutorial_01_quickstart.png"
mean_q0 = model.predict_mean_quantile(X, arm=0).mean(axis=0)
mean_q1 = model.predict_mean_quantile(X, arm=1).mean(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0), constrained_layout=True)
axes[0].plot(u, mean_q0, color="#D55E00", lw=2.0, label="Predicted control barycenter")
axes[0].plot(u, mean_q1, color="#0072B2", lw=2.0, label="Predicted treated barycenter")
axes[0].plot(u, q_star, color="#000000", lw=1.6, ls="--", label="Reference benchmark $q_\star^K$")
axes[0].set_xlabel("Midpoint quantile level $u_k$")
axes[0].set_ylabel("Outcome quantile")
axes[0].set_title("(a) Predicted marginal quantile curves")
axes[0].legend(frameon=False)
axes[0].grid(alpha=0.25)

names = ["mean", "sd", "tail", "reference", "p10", "p90"]
y_pos = np.arange(len(names))
ests = [model.dr_marginal(k) for k in names]
ses = [model.dr_if_se(k) for k in names]
axes[1].axvline(0.0, color="0.3", ls="--", lw=1.0)
axes[1].errorbar(ests, y_pos, xerr=1.96 * np.asarray(ses), fmt="o", color="#0072B2", capsize=4, lw=1.6)
axes[1].set_yticks(y_pos, names)
axes[1].invert_yaxis()
axes[1].set_xlabel("Estimated treatment effect")
axes[1].set_title("(b) Doubly robust functional contrasts")
axes[1].grid(alpha=0.25)

fig.savefig(output_file, dpi=150)
plt.show()

In [ ]:
try:
    from google.colab import files
    files.download(output_file)
    print("Downloaded:", output_file)
except Exception as e:
    print("(Not on Colab / download skipped):", e)